In [0]:
import pyspark.sql.functions as F
df_payroll = spark.read.table("silver_workforce.azure_blob_storage.payroll").alias("payroll")
df_department = spark.read.table("gold_workforce.azure_blob_storage.dim_departments").alias("department")
df_employee = spark.read.table("gold_workforce.azure_blob_storage.dim_employee").alias("employee")
df_company = spark.read.table("gold_workforce.azure_blob_storage.dim_company").alias("company")

#display(df_department)

df_joined = df_payroll.join(df_department, df_payroll.department_id == df_department.department_id, "inner") \
    .join(df_employee, df_payroll.employee_id == df_employee.employee_id, "inner") \
    .join(df_company, df_payroll.company_id == df_company.company_id, "inner").selectExpr(
    'payroll.employee_id',
    'payroll.department_id',
    'payroll.company_id',
    'payroll_id',
    'pay_period_start',
    'pay_period_end',
    'pay_date',
    'bonus',
    'overtime_pay',
    'commission',
    'allowances',
    'tax_deduction',
    'social_security',
    'health_insurance',
    'retirement_contribution',
    'other_deductions',
    'employee.base_salary',
    'net_salary',
    'department_key',
    'termination_date',
    'is_active',
    'employee_key',
    'company_key',

)

display(df_joined)

In [0]:
df_joined.write.format("delta").mode("overwrite").saveAsTable("gold_workforce.azure_blob_storage.fact_payroll")

In [0]:
import pyspark.sql.functions as F
df_general_ledgers = spark.read.table("silver_workforce.azure_blob_storage.general_ledgers").alias("general_ledger")
df_department = spark.read.table("gold_workforce.azure_blob_storage.dim_departments").alias("departments")
df_accounts = spark.read.table("gold_workforce.azure_blob_storage.dim_accounts").alias("accounts")
df_company = spark.read.table("gold_workforce.azure_blob_storage.dim_company").alias("company")

display(df_general_ledgers)

In [0]:
df_joined = df_general_ledgers.join(df_department, df_general_ledgers.department_id == df_department.department_id, "inner") \
    .join(df_accounts, df_general_ledgers.account_id == df_accounts.account_id, "inner") \
    .join(df_company, df_general_ledgers.company_id == df_company.company_id, "inner").selectExpr(
    'gl_id',
    'entry_date',
    'posting_date',
    'fiscal_year',
    'fiscal_period',
    'company.company_id',
    'accounts.account_id',
    'departments.department_id',
    'debit_amount',
    'credit_amount',
    'created_by',
    'approved_by',
    'department_key',
    'account_key',
    'company_key'
)

display(df_joined)

In [0]:
df_joined = df_joined.withColumn("total_amount", F.col("credit_amount") + F.col("debit_amount"))

display(df_joined)

In [0]:
df_joined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_workforce.azure_blob_storage.fact_general_ledgers")